In [ ]:
import openai
from arguments import get_config
from interfaces import setup_LMP
from visualizers import ValueMapVisualizer
from envs.rlbench_env import VoxPoserRLBench
from utils import set_lmp_objects
import numpy as np
from rlbench import tasks

openai.api_key = 'OpenAI API KEY'  # set your API key here

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Setup

In [2]:
config = get_config('rlbench')
# uncomment this if you'd like to change the language model (e.g., for faster speed or lower cost)
# for lmp_name, cfg in config['lmp_config']['lmps'].items():
#     cfg['model'] = 'gpt-3.5-turbo'

# initialize env and voxposer ui
visualizer = ValueMapVisualizer(config['visualizer'])
env = VoxPoserRLBench(visualizer=visualizer)
lmps, lmp_env = setup_LMP(env, config, debug=False)
voxposer_ui = lmps['plan_ui']

##################################################
## voxel resolution: [0.0105 0.0131 0.01  ]
##################################################




## Playground

By default we use one of the instructions that come with each task. However, you may treat each task as simply a scene initialization from RLBench, and feel free to try any task you can come up with for the scene.

Note:
- Whether an instruction can be executed or not depends on 1) whether relevant objects are available, and 2) capabilities of the overall algorithm.
- Each execution may produce one or more visualizations. You may view them in "./visualizations/" folder.
- The prompts are adapted with minimal change from the real-world environment in the VoxPoser paper. If a task failure is due to incorrect code generated by the LLM, feel free to modify the relevant prompt in "./prompts/" folder.
- You may view the reward by printing "env.latest_reward". These are computed by RLBench for each task.
- To inspect in viewer without performing any action, you may call "env.rlbench_env._scene.step()" in a loop.

In [3]:
# # uncomment this to show all available tasks in rlbench
# # NOTE: in order to run a new task, you need to add the list of objects (and their corresponding env names) to the "task_object_names.json" file. See README for more details.
# print([task for task in dir(tasks) if task[0].isupper() and not '_' in task])

In [4]:
# below are the tasks that have object names added to the "task_object_names" file
# uncomment one to use

# task_name = "PutRubbishInBin"
# env.load_task(tasks.PutRubbishInBin)

# task_name = "LampOff"
# env.load_task(tasks.LampOff)

# task_name = "OpenWineBottle"
# env.load_task(tasks.OpenWineBottle)

task_name = "PushButton"
env.load_task(tasks.PushButton)

# task_name = "TakeOffWeighingScales"
# env.load_task(tasks.TakeOffWeighingScales)

# task_name = "MeatOffGrill"
# env.load_task(tasks.MeatOffGrill)

# task_name = "SlideBlockToTarget"
# env.load_task(tasks.SlideBlockToTarget)

# task_name = "TakeLidOffSaucepan"
# env.load_task(tasks.TakeLidOffSaucepan)

# task_name = "TakeUmbrellaOutOfUmbrellaStand"
# env.load_task(tasks.TakeUmbrellaOutOfUmbrellaStand)

descriptions, obs = env.reset()
set_lmp_objects(lmps, env.get_object_names())  # set the object names

In [6]:
import os
import sys
from datetime import datetime

# create log directory for the current task
log_root = "logs"
task_log_dir = os.path.join(log_root, task_name)
os.makedirs(task_log_dir, exist_ok=True)

# log filename for the current run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = os.path.join(task_log_dir, f"{task_name}_{timestamp}.txt")

class Tee:
    def __init__(self, filepath):
        self.stdout = sys.stdout
        self.file = open(filepath, "w", encoding="utf-8")

    def write(self, data):
        self.stdout.write(data)
        self.file.write(data)

    def flush(self):
        self.stdout.flush()
        self.file.flush()

    def close(self):
        self.file.close()

# restore original stdout to avoid double-wrapping
if "_original_stdout" not in globals():
    _original_stdout = sys.stdout
else:
    sys.stdout = _original_stdout

logger = Tee(log_file)
sys.stdout = logger

print("=" * 80)
print(f"Task: {task_name}")
print(f"Log file: {log_file}")
print(f"Start time: {timestamp}")
print("=" * 80)

Task: PushButton
Log file: logs/PushButton/PushButton_20260325_211622.txt
Start time: 20260325_211622


Instruction: push the cyan button
--------------------------------------------------------------------------------
*** OpenAI API call took 1.14s ***
########################################
## "planner" generated code
## context: "objects = ['button']"
########################################
objects = ['button']
# Query: push the cyan button.
composer("push the cyan button")


*** OpenAI API call took 1.48s ***
########################################
## "composer" generated code
########################################
# Query: push the cyan button.
movable = parse_query_obj('gripper')
affordance_map = get_affordance_map('the cyan button')
execute(movable, affordance_map=affordance_map)


*** OpenAI API call took 1.11s ***
########################################
## "parse_query_obj" generated code
## context: "objects = ['button']"
########################################
objects = ['button']
# Query: gripper.
gripper = detect('gripper')
ret_val = gripper


*** OpenAI API call took

In [7]:
instruction = np.random.choice(descriptions)
print(f"Instruction: {instruction}")
print("-" * 80)

voxposer_ui(instruction)
reward = env.latest_reward
print(f"Reward: {reward}") 
print("\n" + "=" * 80)
print("Run finished.")
print("=" * 80)

# automatically save successful episode to memory
if reward == 1.0:
    try:
        import sys, os
        sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
        from memory.log_parser import parse_log
        from memory.memory_store import add_episode
        ep = parse_log(log_file)
        add_episode(ep)
        print("[memory] episode saved to memory automatically")
    except Exception as e:
        print(f"[memory] failed to save episode: {e}")